# NN-kNN Classification Workflow

This notebook uses the maintained IJCAI-26 NN-kNN core for classification. Retrieval, feature weighting, case scoring, and case normalization remain the current model; the output layer sums normalized case activation into class probability mass and trains it with negative log likelihood.

In [ ]:
from pathlib import Path
import os
import sys

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "model").is_dir() and (candidate / "datasets").is_dir():
            return candidate
    raise RuntimeError("Could not find NN-kNN repo root from this notebook.")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from model.classification_workflow import (
    list_supported_classification_datasets,
    list_supported_classification_benchmark_methods,
    make_classification_cfg,
    run_single_nnknn_classification_experiment,
    run_repeated_classification_model_benchmarks,
)

print(list_supported_classification_datasets())
print(list_supported_classification_benchmark_methods())

## Small Dataset Sanity Check

The IJCAI-25 small-data family is available as `iris`, `zebra`, `zebra_special`, `wine`, `breast_cancer`, `balance`, and `digits`. Standardization is fitted on the training split only.

Iris-
Epochs: 50.0,
Tau: 0.001

Zebra-
Epochs: 550,
Tau: 100.0 

Zebra_special-
Epochs: 150,
Tau: 1.0

Wine-
Epochs: 50.0, 
Tau: 0.001

breast_cancer-
Epochs: 50,0
Tau: 100,0

balance-
Epochs: 50.0,
Tau:.01

digits-
Epochs: 100.0.
Tau: 0.001

In [ ]:
dataset = "iris"  # Try "zebra", "iris", "wine", "breast_cancer", etc.
dataset_slug = dataset.replace(" ", "_").replace("-", "_")

cfg = make_classification_cfg({
    "training_epochs": 550,
    "batch_size": 32,
    "patience": 40,
    "tau": 100.0,
    "case_normalizer": "softmax",  # Change to "sparsemax" for sparse case activation.
    "top_k": 5,
    "explanation_mode": True,
    "checkpoint_path": "checkpoints/nnknn_classification_notebook.pth",
})

result = run_single_nnknn_classification_experiment(
    dataset, cfg, run_seed=42, split_seed=42, checkpoint_label="digits_demo"
)
print("Validation accuracy:", result["accuracy"])
print("Auto-standardized features:", result["standardized_features"])
print("First class probability masses:", result["class_probabilities"][:3])

In [ ]:
# Explanation output uses cases retrieved by the current model.
print("Top retrieved class ids for query 0:", result["most_activated_class_ids"][0])
print("Top retrieved activation mass for query 0:", result["most_activated_activations"][0])

In [ ]:
import numpy as np
from ipywidgets import interact, IntSlider

import matplotlib.pyplot as plt

def plot_query_with_cases(query_idx):
    """Plot a single query and its top-k activated cases on 2D scatter plot."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Get all training data
    X_train = result["X_train"].numpy()
    y_train = result["y_train"].numpy()
    
    # Plot all training cases, color-coded by class
    colors = ['blue', 'red']
    for class_id in range(result["num_classes"]):
        mask = y_train == class_id
        ax.scatter(X_train[mask, 0], X_train[mask, 1], 
                  c=colors[class_id], label=f'Class {class_id}', 
                  alpha=0.5, s=50, edgecolors='black', linewidth=0.5)
    
    # Get query point
    query_point = result["X_val"][query_idx].numpy()
    ax.scatter(query_point[0], query_point[1], 
              c='green', marker='*', s=500, label='Query', 
              edgecolors='black', linewidth=1.5, zorder=5)
    
    # Get top-k activated cases and their activations
    top_cases = result["most_activated_cases"][query_idx].numpy()
    top_class_ids = result["most_activated_class_ids"][query_idx].numpy()
    top_activations = result["most_activated_activations"][query_idx].numpy()
    
    # Plot top-k cases with size proportional to activation
    for k, (case, class_id, activation) in enumerate(zip(top_cases, top_class_ids, top_activations)):
        size = 200 * activation  # Scale size by activation strength
        ax.scatter(case[0], case[1], c=colors[class_id], marker='s', 
                  s=size, alpha=0.7, edgecolors='gold', linewidth=2, 
                  label=f'Top-{k+1} (act={activation:.3f})', zorder=4)
    
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.set_title(f'Query {query_idx} - Predicted Class: {result["predictions"][query_idx].item()}')
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(plot_query_with_cases, query_idx=IntSlider(min=0, max=10, step=1, value=0))

In [ ]:
glocal = result.get("glocal_weightor", result["model"].glocal_weightor)

if hasattr(glocal, "state_dict"):
    for name, value in glocal.state_dict().items():
        print(f"{name}: {value}")
elif hasattr(glocal, "weights"):
    print("weights:", glocal.weights)
else:
    print(glocal)

## Representative Benchmark Check

This short run checks that NN-kNN, kNN, and a four-hidden-layer MLP execute on shared stratified folds. Use more folds and epochs for a table intended for reporting.

In [ ]:
summary, runs, _ = run_repeated_classification_model_benchmarks(
    dataset,
    cfg,
    methods=["nnknn", "knn", "mlp"],
    num_runs=3,
    mode="kfold",
    base_seed=42,
    method_cfgs={"mlp": {"epochs": 50, "patience": 10}},
)
summary

## Image Workflow

For `mnist`, `cifar10`, or `svhn`, the loader uses the official train/test split and fits normalization statistics on training images. Training reserves an inner validation slice for checkpoint selection and reports on official test data. Start with a subset before a full image benchmark.

In [ ]:
run_image_demo = False
if run_image_demo:
    image_cfg = make_classification_cfg({
        "training_epochs": 5,
        "batch_size": 64,
        "top_k": 5,
        "explanation_mode": True,
        "checkpoint_path": "checkpoints/nnknn_mnist_subset.pth",
    })
    image_result = run_single_nnknn_classification_experiment(
        "mnist",
        image_cfg,
        dataset_kwargs={"max_train_samples": 1000, "max_eval_samples": 300},
        checkpoint_label="mnist_subset",
    )
    print("MNIST subset accuracy:", image_result["accuracy"])

For image baseline comparisons, call `run_repeated_classification_model_benchmarks` with methods `convnet`, `knn_pixels`, `knn_conv_frozen`, `nnknn_conv_trainable`, and `nnknn_conv_frozen`. The frozen NN-kNN method reuses a trained ConvNet feature extractor and keeps it frozen during NN-kNN training.

In [ ]:
import itertools
import numpy as np
import pandas as pd

run_hyperparameter_tuning = True

tuning_dataset = dataset
tuning_dataset_slug = globals().get(
    "dataset_slug",
    tuning_dataset.replace(" ", "_").replace("-", "_")
)

hyperparameter_grid = {
    "training_epochs": [100, 200, 300, 400],
    "tau": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],
}

def extract_feature_weights(run_result):
    glocal = run_result.get("glocal_weightor", run_result["model"].glocal_weightor)

    if hasattr(glocal, "get_feature_weights_display"):
        weights = glocal.get_feature_weights_display()
    elif hasattr(glocal, "weights"):
        weights = glocal.weights
    elif hasattr(glocal, "state_dict"):
        state = glocal.state_dict()
        weights = next(iter(state.values()))
    else:
        return None

    if hasattr(weights, "detach"):
        weights = weights.detach().cpu().numpy()
    elif hasattr(weights, "cpu"):
        weights = weights.cpu().numpy()

    weights = np.asarray(weights).reshape(-1)
    return weights.round(4).tolist()

hyperparameter_tuning_results = pd.DataFrame()
best_tuning_cfg = None

if run_hyperparameter_tuning:
    tuning_records = []
    grid_keys = list(hyperparameter_grid)
    grid_values = [hyperparameter_grid[key] for key in grid_keys]
    grid_size = 1
    for values_for_key in grid_values:
        grid_size *= len(values_for_key)

    for run_number, values in enumerate(itertools.product(*grid_values), start=1):
        params = dict(zip(grid_keys, values))
        label = "_".join(f"{key}_{value}" for key, value in params.items())

        print(f"[{run_number}/{grid_size}] {tuning_dataset}: {params}")

        tuning_cfg = make_classification_cfg(
            {
                **cfg,
                **params,
                "checkpoint_path": (
                    f"checkpoints/nnknn_{tuning_dataset_slug}_{label}.pth"
                ),
            }
        )

        tuning_result = run_single_nnknn_classification_experiment(
            tuning_dataset,
            tuning_cfg,
            run_seed=42,
            split_seed=42,
            checkpoint_label=f"{tuning_dataset_slug}_{label}",
        )

        probabilities = tuning_result["class_probabilities"].detach().cpu()
        confidence = probabilities.max(dim=1).values.mean().item()
        feature_weights = extract_feature_weights(tuning_result)

        record = {
            **params,
            "accuracy": tuning_result["accuracy"],
            "mean_confidence": confidence,
            "feature_weights": feature_weights,
            "checkpoint_path": tuning_cfg["checkpoint_path"],
        }

        if feature_weights is not None:
            for feature_idx, weight in enumerate(feature_weights):
                record[f"feature_weight_{feature_idx}"] = weight

        tuning_records.append(record)

    hyperparameter_tuning_results = pd.DataFrame(tuning_records).sort_values(
        ["accuracy", "mean_confidence"],
        ascending=[False, False],
    )

    best_params = {
        key: hyperparameter_tuning_results.iloc[0][key]
        for key in hyperparameter_grid
    }
    best_tuning_cfg = make_classification_cfg(
        {
            **cfg,
            **best_params,
            "checkpoint_path": hyperparameter_tuning_results.iloc[0]["checkpoint_path"],
        }
    )

    display(hyperparameter_tuning_results)
    print("Best parameters:", best_params)
else:
    print("Set run_hyperparameter_tuning = True to run the grid search.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

inspection_dataset = dataset
inspection_dataset_slug = globals().get(
    "dataset_slug",
    inspection_dataset.replace(" ", "_").replace("-", "_")
)

inspection_result = run_single_nnknn_classification_experiment(
    inspection_dataset,
    cfg,
    run_seed=42,
    split_seed=42,
    checkpoint_label=f"{inspection_dataset_slug}_inspection",
)

probs = inspection_result["class_probabilities"].detach().cpu()
pred = inspection_result["predictions"].detach().cpu()
inspection_true = inspection_result["y_val"].detach().cpu()
inspection_confidence, _ = probs.max(dim=1)

inspection_view = pd.DataFrame(
    {
        "row_id": inspection_result["val_idx"].detach().cpu().numpy(),
        "true_class": inspection_true.numpy(),
        "predicted_class": pred.numpy(),
        "confidence": inspection_confidence.numpy().round(4),
        "correct": (pred == inspection_true).numpy(),
    }
)

for class_idx in range(probs.shape[1]):
    inspection_view[f"p_class_{class_idx}"] = probs[:, class_idx].numpy().round(4)

print(f"{inspection_dataset} accuracy:", inspection_result["accuracy"])
print("Auto-standardized features:", inspection_result["standardized_features"])

display(
    inspection_view.sort_values(["correct", "confidence"], ascending=[True, False])
    .reset_index(drop=True)
)

display(
    pd.crosstab(
        inspection_view["true_class"],
        inspection_view["predicted_class"],
        rownames=["true"],
        colnames=["predicted"],
        dropna=False,
    )
)

raw_X = inspection_result["X"].detach().cpu().numpy()
train_X = inspection_result["X_train_raw"].detach().cpu().numpy()
val_X = inspection_result["X_val_raw"].detach().cpu().numpy()
train_y = inspection_result["y_train"].detach().cpu().numpy()
correct_mask = (pred == inspection_true).numpy()

if raw_X.shape[1] < 2:
    raise ValueError(
        f"{inspection_dataset} has fewer than 2 features, so a 2D map cannot be drawn."
    )

x_pad = 0.05 * max(np.ptp(train_X[:, 0]), 1.0)
y_pad = 0.05 * max(np.ptp(train_X[:, 1]), 1.0)

x_min, x_max = train_X[:, 0].min() - x_pad, train_X[:, 0].max() + x_pad
y_min, y_max = train_X[:, 1].min() - y_pad, train_X[:, 1].max() + y_pad

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 250),
    np.linspace(y_min, y_max, 250),
)

# Full-dimensional grid: features 0 and 1 vary, all other features stay at
# their training-set mean.
grid_raw = np.tile(train_X.mean(axis=0), (xx.size, 1))
grid_raw[:, 0] = xx.ravel()
grid_raw[:, 1] = yy.ravel()

grid_features = (
    inspection_result["scaler"].transform(grid_raw)
    if inspection_result["scaler"] is not None
    else grid_raw
)

model = inspection_result["model"]
device = next(model.parameters()).device
model.eval()

grid_predictions = []
with torch.no_grad():
    for start in range(0, len(grid_features), 2048):
        grid_tensor = torch.tensor(
            grid_features[start : start + 2048],
            dtype=torch.float32,
            device=device,
        )
        _, predicted_grid, *_ = model(grid_tensor)
        grid_predictions.append(predicted_grid.detach().cpu())

grid_predictions = torch.cat(grid_predictions).numpy().reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(
    xx,
    yy,
    grid_predictions,
    levels=np.arange(probs.shape[1] + 1) - 0.5,
    cmap="Pastel1",
    alpha=0.65,
)
ax.scatter(
    train_X[:, 0],
    train_X[:, 1],
    c=train_y,
    cmap="Set1",
    marker=".",
    alpha=0.35,
    label="train true class",
)
ax.scatter(
    val_X[:, 0],
    val_X[:, 1],
    c=pred.numpy(),
    cmap="Set1",
    edgecolor="black",
    linewidth=0.8,
    s=70,
    label="validation predicted class",
)
ax.scatter(
    val_X[~correct_mask, 0],
    val_X[~correct_mask, 1],
    facecolors="none",
    edgecolors="red",
    linewidth=2.2,
    s=160,
    label="misclassified",
)
ax.set_title(f"NN-kNN classification map: {inspection_dataset} features 0-1 slice")
ax.set_xlabel("feature 0")
ax.set_ylabel("feature 1")
ax.legend(loc="upper right")
plt.show()

if "most_activated_class_ids" in inspection_result:
    missed_positions = inspection_view.index[~inspection_view["correct"]].tolist()[:5]
    retrieved_classes = inspection_result["most_activated_class_ids"].detach().cpu().numpy()
    retrieved_activations = inspection_result["most_activated_activations"].detach().cpu().numpy()

    for pos in missed_positions:
        print(
            f"Validation position {pos}, original row {inspection_view.loc[pos, 'row_id']}: "
            f"true={inspection_view.loc[pos, 'true_class']}, "
            f"predicted={inspection_view.loc[pos, 'predicted_class']}"
        )
        print("Retrieved class ids:", retrieved_classes[pos].tolist())
        print("Retrieved activation mass:", retrieved_activations[pos].round(4).tolist())

In [ ]:
all_rows = pd.DataFrame(
    {
        "row_id": np.arange(len(raw_X)),
        "feature_0": raw_X[:, 0],
        "feature_1": raw_X[:, 1],
        "true_class": inspection_result["y"].detach().cpu().numpy(),
    }
)

validation_details = inspection_view[
    ["row_id", "predicted_class", "confidence", "correct"]
    + [column for column in inspection_view.columns if column.startswith("p_class_")]
]
all_rows = all_rows.merge(validation_details, on="row_id", how="left")
all_rows["split"] = np.where(all_rows["predicted_class"].notna(), "validation", "train")

ordered_columns = [
    "row_id",
    "split",
    "feature_0",
    "feature_1",
    "true_class",
    "predicted_class",
    "confidence",
    "correct",
] + [column for column in all_rows.columns if column.startswith("p_class_")]
all_rows = all_rows[ordered_columns]

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(all_rows)

print(all_rows.to_string(index=False))

In [ ]:
validation_only = all_rows[all_rows["predicted_class"].notna()]

print(validation_only)

In [ ]:
import matplotlib.pyplot as plt

validation_only = all_rows[all_rows["predicted_class"].notna()].copy()
misses = validation_only[validation_only["correct"] == False].copy()

print(f"Validation rows: {len(validation_only)}")
print(f"Misclassified rows: {len(misses)}")
display(misses)

train_rows = inspection_result["train_idx"].detach().cpu().numpy()
train_raw = inspection_result["X_train_raw"].detach().cpu().numpy()
train_labels = inspection_result["y_train"].detach().cpu().numpy()
val_rows = inspection_result["val_idx"].detach().cpu().numpy()
val_raw = inspection_result["X_val_raw"].detach().cpu().numpy()
retrieved_cases = inspection_result.get("most_activated_cases")
retrieved_class_ids = inspection_result.get("most_activated_class_ids")
retrieved_activations = inspection_result.get("most_activated_activations")

if retrieved_cases is not None:
    retrieved_cases = retrieved_cases.detach().cpu().numpy()
    retrieved_class_ids = retrieved_class_ids.detach().cpu().numpy()
    retrieved_activations = retrieved_activations.detach().cpu().numpy()

def build_retrieved_table(val_position):
    if retrieved_cases is None:
        return pd.DataFrame()
    retrieved_for_query = retrieved_cases[val_position]
    if np.issubdtype(retrieved_for_query.dtype, np.integer):
        retrieved_positions = retrieved_for_query.astype(int)
        retrieved_row_ids = train_rows[retrieved_positions]
        retrieved_features = train_raw[retrieved_positions]
    else:
        retrieved_features = retrieved_for_query.reshape(retrieved_for_query.shape[0], -1)
        nearest_matches = np.argmin(
            np.linalg.norm(train_raw[:, None, :] - retrieved_features[None, :, :], axis=2),
            axis=0,
        )
        retrieved_positions = nearest_matches.astype(int)
        retrieved_row_ids = train_rows[retrieved_positions]

    return pd.DataFrame(
        {
            "retrieved_train_position": retrieved_positions,
            "retrieved_row_id": retrieved_row_ids,
            "retrieved_class": retrieved_class_ids[val_position],
            "activation_mass": retrieved_activations[val_position].round(4),
            "retrieved_feature_0": retrieved_features[:, 0],
            "retrieved_feature_1": retrieved_features[:, 1],
        }
    )

for _, miss in misses.iterrows():
    row_id = int(miss["row_id"])
    val_position = int(np.where(val_rows == row_id)[0][0])
    query = val_raw[val_position]
    distances = np.linalg.norm(train_raw - query, axis=1)
    nearest_positions = np.argsort(distances)[:10]
    nearest = pd.DataFrame(
        {
            "train_row_id": train_rows[nearest_positions],
            "feature_0": train_raw[nearest_positions, 0],
            "feature_1": train_raw[nearest_positions, 1],
            "true_class": train_labels[nearest_positions],
            "raw_distance": distances[nearest_positions].round(4),
        }
    )
    retrieved = build_retrieved_table(val_position)

    prob_columns = [column for column in all_rows.columns if column.startswith("p_class_")]
    probabilities = miss[prob_columns].astype(float).sort_values(ascending=False)
    margin = probabilities.iloc[0] - probabilities.iloc[1] if len(probabilities) > 1 else np.nan
    true_class = int(miss["true_class"])
    predicted_class = int(miss["predicted_class"])

    print("\n" + "=" * 80)
    print(
        f"row_id={row_id} | features=({miss['feature_0']:.3f}, {miss['feature_1']:.3f}) | "
        f"true={true_class} | predicted={predicted_class} | "
        f"confidence={miss['confidence']:.4f} | margin={margin:.4f}"
    )

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(
        train_raw[:, 0],
        train_raw[:, 1],
        c=train_labels,
        cmap="Set1",
        marker=".",
        alpha=0.22,
        s=35,
        label="all training rows",
    )
    ax.scatter(
        train_raw[nearest_positions, 0],
        train_raw[nearest_positions, 1],
        c=train_labels[nearest_positions],
        cmap="Set1",
        edgecolor="black",
        linewidth=0.8,
        s=95,
        label="10 nearest raw rows",
    )
    if not retrieved.empty:
        activation_sizes = 220 + 1200 * retrieved["activation_mass"].to_numpy(dtype=float)
        ax.scatter(
            retrieved["retrieved_feature_0"],
            retrieved["retrieved_feature_1"],
            c=retrieved["retrieved_class"],
            cmap="Set1",
            marker="s",
            edgecolor="gold",
            linewidth=2.2,
            s=activation_sizes,
            alpha=0.75,
            label="NN-kNN retrieved cases",
        )
        for _, retrieved_row in retrieved.iterrows():
            ax.annotate(
                f"{int(retrieved_row['retrieved_row_id'])}\n{retrieved_row['activation_mass']:.2f}",
                (retrieved_row["retrieved_feature_0"], retrieved_row["retrieved_feature_1"]),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=8,
            )
    ax.scatter(
        [query[0]],
        [query[1]],
        marker="X",
        c=[true_class],
        cmap="Set1",
        edgecolor="black",
        linewidth=1.6,
        s=280,
        label="misclassified query",
    )
    ax.annotate(f"query row {row_id}", (query[0], query[1]), xytext=(8, -14), textcoords="offset points")
    ax.set_title(
        f"Misclassified {inspection_dataset} row {row_id}: true={true_class}, predicted={predicted_class}"
    )
    ax.set_xlabel("feature 0")
    ax.set_ylabel("feature 1")
    ax.legend(loc="best")
    plt.show()

    print("Class probabilities:")
    print(probabilities.to_string())
    print("\nNearest raw training points:")
    display(nearest)

    if not retrieved.empty:
        print("\nNN-kNN retrieved/activated cases:")
        display(retrieved)
        print("Retrieved class vote totals:")
        display(retrieved.groupby("retrieved_class")["activation_mass"].sum().sort_values(ascending=False))

In [ ]:
epoch_grid = [20, 30, 40, 50, 60, 75]
baseline_epoch = epoch_grid[0]
epoch_results = {}

inspection_dataset_slug = globals().get(
    "dataset_slug",
    inspection_dataset.replace(" ", "_").replace("-", "_")
)

for epochs in epoch_grid:
    cfg_epoch = make_classification_cfg(
        {
            **cfg,
            "training_epochs": epochs,
            "checkpoint_path": f"checkpoints/nnknn_{inspection_dataset_slug}_epochs_{epochs}.pth",
        }
    )
    result_epoch = run_single_nnknn_classification_experiment(
        inspection_dataset,
        cfg_epoch,
        run_seed=42,
        split_seed=42,
        checkpoint_label=f"{inspection_dataset_slug}_epochs_{epochs}",
    )
    epoch_results[epochs] = result_epoch

baseline_result = epoch_results[baseline_epoch]
baseline_pred = baseline_result["predictions"].detach().cpu()
baseline_true = baseline_result["y_val"].detach().cpu()
baseline_rows = baseline_result["val_idx"].detach().cpu().numpy()
baseline_probs = baseline_result["class_probabilities"].detach().cpu()
baseline_confidence = baseline_probs.max(dim=1).values

baseline_miss_positions = (
    (baseline_pred != baseline_true)
    .nonzero(as_tuple=False)
    .flatten()
    .tolist()
)
baseline_missed_row_ids = [
    int(baseline_rows[position])
    for position in baseline_miss_positions
]

if baseline_missed_row_ids:
    tracked_row_ids = baseline_missed_row_ids
    tracking_reason = f"rows misclassified at baseline epoch {baseline_epoch}"
else:
    fallback_count = min(5, len(baseline_rows))
    least_confident_positions = torch.argsort(baseline_confidence)[:fallback_count].tolist()
    tracked_row_ids = [
        int(baseline_rows[position])
        for position in least_confident_positions
    ]
    tracking_reason = (
        f"least-confident rows at baseline epoch {baseline_epoch} "
        "because there were no baseline misses"
    )

print(f"Tracking {tracking_reason}:")
print(tracked_row_ids)

tracked_vote_rows = []

for epochs, result_epoch in epoch_results.items():
    probs_epoch = result_epoch["class_probabilities"].detach().cpu()
    pred_epoch = result_epoch["predictions"].detach().cpu()
    true_epoch = result_epoch["y_val"].detach().cpu()
    val_rows_epoch = result_epoch["val_idx"].detach().cpu().numpy()
    confidence_epoch, _ = probs_epoch.max(dim=1)

    retrieved_classes_epoch = result_epoch.get("most_activated_class_ids")
    retrieved_activations_epoch = result_epoch.get("most_activated_activations")

    retrieved_votes_available = (
        retrieved_classes_epoch is not None
        and retrieved_activations_epoch is not None
    )
    if retrieved_votes_available:
        retrieved_classes_epoch = retrieved_classes_epoch.detach().cpu().numpy()
        retrieved_activations_epoch = retrieved_activations_epoch.detach().cpu().numpy()

    print("\n" + "=" * 88)
    print(f"epochs={epochs} | accuracy={result_epoch['accuracy']:.4f}")

    for row_id in tracked_row_ids:
        matching_positions = np.where(val_rows_epoch == row_id)[0]
        if len(matching_positions) == 0:
            tracked_vote_rows.append(
                {
                    "epochs": epochs,
                    "baseline_epoch": baseline_epoch,
                    "row_id": row_id,
                    "status": "not_in_validation_split",
                }
            )
            continue

        val_position = int(matching_positions[0])
        probability_values = probs_epoch[val_position].numpy()
        sorted_probabilities = np.sort(probability_values)
        probability_margin = (
            sorted_probabilities[-1] - sorted_probabilities[-2]
            if len(sorted_probabilities) >= 2
            else np.nan
        )

        true_class = int(true_epoch[val_position])
        predicted_class = int(pred_epoch[val_position])
        is_correct = predicted_class == true_class

        row = {
            "epochs": epochs,
            "baseline_epoch": baseline_epoch,
            "row_id": row_id,
            "status": "correct_now" if is_correct else "still_misclassified",
            "true_class": true_class,
            "predicted_class": predicted_class,
            "confidence": float(confidence_epoch[val_position]),
            "probability_margin": float(probability_margin),
        }

        if retrieved_votes_available:
            class_votes = (
                pd.DataFrame(
                    {
                        "retrieved_class": retrieved_classes_epoch[val_position],
                        "activation_mass": retrieved_activations_epoch[val_position],
                    }
                )
                .groupby("retrieved_class")["activation_mass"]
                .sum()
                .sort_index()
            )

            row["retrieved_vote_winner"] = int(class_votes.idxmax())
            row["true_class_retrieved_mass"] = float(class_votes.get(true_class, 0.0))
            row["predicted_class_retrieved_mass"] = float(
                class_votes.get(predicted_class, 0.0)
            )

            for retrieved_class, activation_mass in class_votes.items():
                row[f"retrieved_class_{int(retrieved_class)}_mass"] = float(
                    activation_mass
                )
        else:
            class_votes = pd.Series(dtype="float32", name="activation_mass")
            row["retrieved_vote_winner"] = np.nan
            row["true_class_retrieved_mass"] = np.nan
            row["predicted_class_retrieved_mass"] = np.nan

        tracked_vote_rows.append(row)

        print(
            f"\nrow_id={row_id} | {row['status']} | true={true_class} | "
            f"predicted={predicted_class} | confidence={row['confidence']:.4f} | "
            f"probability_margin={probability_margin:.4f}"
        )
        print(class_votes)

tracked_epoch_vote_summary = pd.DataFrame(tracked_vote_rows)

print(
    f"Expected tracked rows: {len(tracked_row_ids)} tracked rows "
    f"x {len(epoch_grid)} epochs = {len(tracked_row_ids) * len(epoch_grid)}"
)
print(f"Actual tracked rows: {len(tracked_epoch_vote_summary)}")

if tracked_epoch_vote_summary.empty:
    print("No rows were tracked.")
else:
    display(
        tracked_epoch_vote_summary
        .sort_values(["row_id", "epochs"])
        .reset_index(drop=True)
    )

## Epoch failure visualizations

In [ ]:
plot_row_id = 52
row_trace = (
    tracked_epoch_vote_summary[tracked_epoch_vote_summary["row_id"] == plot_row_id]
    .sort_values("epochs")
    .copy()
)

if row_trace.empty:
    raise ValueError(f"row_id {plot_row_id} was not tracked. Check baseline_missed_row_ids above.")

mass_columns = sorted(
    [column for column in row_trace.columns if column.startswith("retrieved_class_") and column.endswith("_mass")],
    key=lambda column: int(column.split("_")[2]),
)

fig, ax = plt.subplots(figsize=(9, 5.5))
colors = {0: "#2f6fbb", 1: "#d14b3f"}

for column in mass_columns:
    class_id = int(column.split("_")[2])
    label = f"retrieved class {class_id} mass"
    ax.plot(
        row_trace["epochs"],
        row_trace[column],
        marker="o",
        linewidth=2.6,
        markersize=7,
        color=colors.get(class_id),
        label=label,
    )

correct_mask = row_trace["status"] == "correct_now"
ax.scatter(
    row_trace.loc[correct_mask, "epochs"],
    row_trace.loc[correct_mask, "true_class_retrieved_mass"],
    marker="*",
    s=220,
    color="#1f8f4d",
    edgecolor="white",
    linewidth=1.2,
    zorder=5,
    label="correct at this epoch",
)
ax.scatter(
    row_trace.loc[~correct_mask, "epochs"],
    row_trace.loc[~correct_mask, "true_class_retrieved_mass"],
    marker="X",
    s=110,
    color="#111111",
    zorder=5,
    label="misclassified at this epoch",
)

for _, row in row_trace.iterrows():
    ax.annotate(
        f"pred {int(row['predicted_class'])}",
        (row["epochs"], row["true_class_retrieved_mass"]),
        xytext=(0, 12),
        textcoords="offset points",
        ha="center",
        fontsize=9,
    )

true_class = int(row_trace["true_class"].dropna().iloc[0])
ax.set_title(f"Retrieved class mass over training for dataset row {plot_row_id}", fontsize=14, pad=14)
ax.set_xlabel("training epochs")
ax.set_ylabel("retrieved activation mass")
ax.set_ylim(-0.03, 1.03)
ax.set_xticks(row_trace["epochs"])
ax.grid(axis="y", alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.text(
    0.01,
    0.96,
    f"true class: {true_class}",
    transform=ax.transAxes,
    fontsize=11,
    va="top",
    bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "#dddddd", "alpha": 0.9},
)
ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.show()

display(row_trace[["epochs", "status", "true_class", "predicted_class", "confidence", "probability_margin"] + mass_columns])

In [ ]:
diagnostic_row_id = plot_row_id
early_epoch = baseline_epoch

row_trace = (
    tracked_epoch_vote_summary[tracked_epoch_vote_summary["row_id"] == diagnostic_row_id]
    .sort_values("epochs")
    .copy()
)
if row_trace.empty:
    raise ValueError(f"row_id {diagnostic_row_id} was not tracked.")

correct_epochs = row_trace.loc[row_trace["status"] == "correct_now", "epochs"]
comparison_epoch = int(correct_epochs.iloc[0]) if len(correct_epochs) else int(row_trace["epochs"].iloc[-1])
print(f"Comparing row {diagnostic_row_id}: early epoch {early_epoch} vs comparison epoch {comparison_epoch}")


def retrieved_table_for_epoch(result_epoch, row_id):
    train_rows_epoch = result_epoch["train_idx"].detach().cpu().numpy()
    train_raw_epoch = result_epoch["X_train_raw"].detach().cpu().numpy()
    train_labels_epoch = result_epoch["y_train"].detach().cpu().numpy()
    val_rows_epoch = result_epoch["val_idx"].detach().cpu().numpy()
    val_raw_epoch = result_epoch["X_val_raw"].detach().cpu().numpy()
    val_position = int(np.where(val_rows_epoch == row_id)[0][0])
    query = val_raw_epoch[val_position]

    retrieved_cases_epoch = result_epoch.get("most_activated_cases")
    retrieved_classes_epoch = result_epoch.get("most_activated_class_ids")
    retrieved_activations_epoch = result_epoch.get("most_activated_activations")
    if retrieved_cases_epoch is None or retrieved_classes_epoch is None or retrieved_activations_epoch is None:
        return pd.DataFrame(), pd.Series(dtype="float32", name="activation_mass"), query, val_position

    retrieved_cases_epoch = retrieved_cases_epoch.detach().cpu().numpy()
    retrieved_classes_epoch = retrieved_classes_epoch.detach().cpu().numpy()
    retrieved_activations_epoch = retrieved_activations_epoch.detach().cpu().numpy()
    retrieved_for_query = retrieved_cases_epoch[val_position]

    if np.issubdtype(retrieved_for_query.dtype, np.integer):
        retrieved_positions = retrieved_for_query.astype(int)
        retrieved_row_ids = train_rows_epoch[retrieved_positions]
        retrieved_features = train_raw_epoch[retrieved_positions]
    else:
        retrieved_features = retrieved_for_query.reshape(retrieved_for_query.shape[0], -1)
        nearest_matches = np.argmin(
            np.linalg.norm(train_raw_epoch[:, None, :] - retrieved_features[None, :, :], axis=2),
            axis=0,
        )
        retrieved_positions = nearest_matches.astype(int)
        retrieved_row_ids = train_rows_epoch[retrieved_positions]

    retrieved = pd.DataFrame(
        {
            "retrieved_train_position": retrieved_positions,
            "retrieved_row_id": retrieved_row_ids,
            "retrieved_class": retrieved_classes_epoch[val_position],
            "activation_mass": retrieved_activations_epoch[val_position].round(4),
            "retrieved_feature_0": retrieved_features[:, 0],
            "retrieved_feature_1": retrieved_features[:, 1],
            "training_true_class": train_labels_epoch[retrieved_positions],
        }
    )
    class_votes = retrieved.groupby("retrieved_class")["activation_mass"].sum().sort_index()
    return retrieved, class_votes, query, val_position


def summary_for_epoch(result_epoch, row_id, epoch):
    pred_epoch = result_epoch["predictions"].detach().cpu()
    true_epoch = result_epoch["y_val"].detach().cpu()
    probs_epoch = result_epoch["class_probabilities"].detach().cpu()
    val_rows_epoch = result_epoch["val_idx"].detach().cpu().numpy()
    val_position = int(np.where(val_rows_epoch == row_id)[0][0])
    probability_values = probs_epoch[val_position].numpy()
    true_class = int(true_epoch[val_position])
    predicted_class = int(pred_epoch[val_position])
    retrieved, class_votes, query, _ = retrieved_table_for_epoch(result_epoch, row_id)
    return {
        "epochs": epoch,
        "status": "correct_now" if true_class == predicted_class else "still_misclassified",
        "true_class": true_class,
        "predicted_class": predicted_class,
        "confidence": float(probability_values.max()),
        "probability_margin": float(np.sort(probability_values)[-1] - np.sort(probability_values)[-2]),
        "true_class_retrieved_mass": float(class_votes.get(true_class, 0.0)),
        "predicted_class_retrieved_mass": float(class_votes.get(predicted_class, 0.0)),
    }, retrieved, class_votes, query

comparison_rows = []
retrieved_by_epoch = {}
query_by_epoch = {}
for epoch in [early_epoch, comparison_epoch]:
    summary_row, retrieved, class_votes, query = summary_for_epoch(epoch_results[epoch], diagnostic_row_id, epoch)
    comparison_rows.append(summary_row)
    retrieved_by_epoch[epoch] = retrieved
    query_by_epoch[epoch] = query
    print("\n" + "=" * 80)
    print(f"epochs={epoch} | {summary_row['status']} | true={summary_row['true_class']} | predicted={summary_row['predicted_class']}")
    print(class_votes)

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

true_class = int(comparison_df["true_class"].iloc[0])
early_true_mass = comparison_df.loc[comparison_df["epochs"] == early_epoch, "true_class_retrieved_mass"].iloc[0]
later_true_mass = comparison_df.loc[comparison_df["epochs"] == comparison_epoch, "true_class_retrieved_mass"].iloc[0]
early_pred_mass = comparison_df.loc[comparison_df["epochs"] == early_epoch, "predicted_class_retrieved_mass"].iloc[0]
later_pred_mass = comparison_df.loc[comparison_df["epochs"] == comparison_epoch, "predicted_class_retrieved_mass"].iloc[0]

print("\nDiagnosis:")
if early_true_mass < early_pred_mass:
    print("At the early epoch, retrieved activation mass leans toward the predicted class instead of the true class.")
else:
    print("At the early epoch, retrieved mass already favors the true class; the remaining error is likely from probability aggregation or a very small margin.")
if later_true_mass > early_true_mass:
    print(f"True-class retrieved mass rises from {early_true_mass:.3f} to {later_true_mass:.3f}.")
if later_pred_mass < early_pred_mass:
    print(f"Predicted-class retrieved mass falls from {early_pred_mass:.3f} to {later_pred_mass:.3f}.")

train_raw_reference = epoch_results[early_epoch]["X_train_raw"].detach().cpu().numpy()
train_labels_reference = epoch_results[early_epoch]["y_train"].detach().cpu().numpy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5.8), sharex=True, sharey=True)
colors = {0: "#2f6fbb", 1: "#d14b3f"}

for ax, epoch in zip(axes, [early_epoch, comparison_epoch]):
    retrieved = retrieved_by_epoch[epoch]
    query = query_by_epoch[epoch]
    summary_row = comparison_df[comparison_df["epochs"] == epoch].iloc[0]
    ax.scatter(
        train_raw_reference[:, 0],
        train_raw_reference[:, 1],
        c=train_labels_reference,
        cmap="Set1",
        marker=".",
        alpha=0.20,
        s=35,
        label="training rows",
    )
    if not retrieved.empty:
        sizes = 230 + 1300 * retrieved["activation_mass"].to_numpy(dtype=float)
        ax.scatter(
            retrieved["retrieved_feature_0"],
            retrieved["retrieved_feature_1"],
            c=retrieved["retrieved_class"],
            cmap="Set1",
            marker="s",
            edgecolor="gold",
            linewidth=2.1,
            s=sizes,
            alpha=0.78,
            label="retrieved cases",
        )
        for _, retrieved_row in retrieved.iterrows():
            ax.annotate(
                f"{int(retrieved_row['retrieved_row_id'])}\n{retrieved_row['activation_mass']:.2f}",
                (retrieved_row["retrieved_feature_0"], retrieved_row["retrieved_feature_1"]),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=8,
            )
    ax.scatter(
        [query[0]],
        [query[1]],
        marker="X",
        c=[true_class],
        cmap="Set1",
        edgecolor="black",
        linewidth=1.5,
        s=300,
        label="query row",
    )
    ax.set_title(
        f"{epoch} epochs: {summary_row['status']}\n"
        f"pred={int(summary_row['predicted_class'])}, true-mass={summary_row['true_class_retrieved_mass']:.3f}",
        fontsize=12,
    )
    ax.set_xlabel("feature 0")
    ax.grid(alpha=0.18)

axes[0].set_ylabel("feature 1")
axes[1].legend(frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.show()

display(retrieved_by_epoch[early_epoch].assign(epoch=early_epoch))
display(retrieved_by_epoch[comparison_epoch].assign(epoch=comparison_epoch))